In [ ]:
from Datasets.datasets import MyDataset
from Datasets.mapping import country_to_level1, country_to_level2
from Datasets.preprocessing import process_name

In [ ]:

import os
os.environ['NationalityDatasets'] = "./data/"

In [ ]:
dataset = MyDataset()
dataset.load_data(reload=True)
data = dataset.get_data()

In [ ]:
data.shape

In [ ]:
data.sample(n=10)

In [ ]:
data.fillna('', inplace=True)

In [ ]:
data[data.country=='']

In [ ]:
from Datasets.utilities import country_name_standardizer

data.country = data.country.apply(lambda x: country_name_standardizer(x))

In [ ]:
print(data.shape)
data = data[data['country']!='']
print(data.shape)

In [ ]:
data.sample(n=10)

### Preprocessing Name

In [ ]:
data['name'] = data['full-name'].apply(lambda x: process_name(x))

In [ ]:
data[(data.name=='')].sample(n=10)

In [ ]:
data[data.name==''].shape[0]/data.shape[0]

In [ ]:
data[data.country=='Serbia'].sample(n=10)

In [ ]:
# JÃ¼rgen HÃ¼holdt  -> jurgen huholdt
# RomÃ¡n MontaÃ±ez -> roman montanez
# Søren Hess-Olesen -> sren hess olesen
# Mónika Remsei -> monika remsei
# Rupert König -> rupert konig


In [ ]:
data.country.value_counts()[40:50]

In [ ]:
print(data.shape)
data = data[data['name']!='']
print(data.shape)

#### Checking uniqueness!

In [ ]:
data[data.duplicated(subset=['name', 'country'], keep=False)].sample(n=10)

In [ ]:
data[data.name=='martin giuffre']

In [ ]:

data['country'] = data.country.apply(lambda x: 'United States' if x == 'Northern Mariana Islands' else x)

In [ ]:
print(data.shape)
data.drop_duplicates(subset=['name', 'country'], keep='first', inplace=True)
print(data.shape)

In [ ]:
data[data.duplicated(subset=['name'], keep=False)].sample(n=10)

In [ ]:
data[data.name=='russell shorto']

In [ ]:
print(data.shape)
data.drop_duplicates(subset=['name'], inplace=True)
print(data.shape)

#### Checking long names!

In [ ]:
data['length'] = data.name.apply(lambda x: len(x))

In [ ]:
data.length.value_counts()

In [ ]:
data[data.length>50].sample(n=10)

In [ ]:
import matplotlib.pyplot as plt


plt.hist(data.length, bins=100, color='skyblue', edgecolor='black')
plt.xlabel('Name length')
plt.ylabel('Frequency')
plt.title('Distribution of text length')

plt.show()

In [ ]:
data[(data.length>=3) & (data.length<=70)].shape

In [ ]:
data = data[(data.length>=3) & (data.length<=70)]

### Deleting low number countries

In [ ]:
deleted_countries = []
for country, number in zip(data.country.value_counts().index, data.country.value_counts().values):
    if country_to_level2(country) == 'others':
        # if number > 1000:
        #     print(country, number)
        deleted_countries.append(country)
        print(country, number)

In [ ]:
data['level_3'] = data.country
data['level_3'].replace(deleted_countries, 'others', inplace=True)

In [ ]:
data[data['level_3'] == 'others'].sample(n=10)

### Region Level 1 Mapping 

In [ ]:
data['level_1'] = data.country.apply(lambda x: country_to_level1(x))

In [ ]:
print(data.shape)
datasets_path = os.environ['NationalityDatasets']
data[data.level_1=='Deleted'].to_csv(os.path.join(datasets_path, 'deleted_countries.csv'))
data = data[data.level_1!='Deleted']
print(data.shape)

In [ ]:
data[(data.country != '') & (data.level_1 == 'others')].country.value_counts()

In [ ]:
data.level_1.value_counts()

In [ ]:
data.level_1.value_counts().plot.bar()

### Region Level 2 Mapping

In [ ]:
data['level_2'] = data.country.apply(lambda x: country_to_level2(x))

In [ ]:
data[(data.country != '') & (data.level_2 == 'others')].country.value_counts()

In [ ]:
data.level_2.value_counts()

In [ ]:
data.level_2.value_counts().plot.bar()

In [ ]:
data[data.level_2 =='Oceania']

In [ ]:
processed_data = data[['id', 'name', 'level_1', 'level_2', 'level_3', 'country']]

In [ ]:
processed_data = processed_data.sample(frac = 1)

In [ ]:
processed_data

In [ ]:
print(processed_data.shape)
# processed_data = processed_data[processed_data.level_1!='others']
# print(processed_data.shape)
processed_data

In [ ]:
dataset.name = ''
dataset.file_path = './data/'

In [ ]:
dataset.save_data(processed_data, name='data.csv')

In [ ]:
dataset.data = processed_data

In [ ]:
dataset.get_data_path()

In [ ]:
dataset.split_data()

In [ ]:
dataset.save_split_data()

In [ ]:
processed_data.sample(n=10)

In [ ]:
processed_data.level_3.value_counts()

In [ ]:
processed_data[processed_data.level_3 == 'Northern Mariana Islands']

In [ ]:
s1 = set(processed_data.level_3.unique())

In [ ]:
from Datasets.mapping import level2_mapping

s2 = set(level2_mapping.keys())

In [ ]:
s1.difference(s2)

In [ ]:
s2.difference(s1)

In [ ]:
dataset.test